# Sections 1 and 2: Positioning and Direct Replication

ECON 5200 Midterm Project · Replication Study

> **🛑 AI policy, and it is not the problem sets' policy.** Sections 1 and 2 — the
> positioning and the direct replication — are **no AI**. Not for the code, not for
> the prose. Making a computer agree with a published table is the exercise that
> establishes you can read a paper, and a model that has already read the paper will
> simply hand you the answer without your ever finding the discrepancy. Sections 3 to
> 6 authorise AI as a documented co-pilot under **P.R.I.M.E.**, on the same terms as
> every problem set: you file the full transcript in `ai-appendix.md`, and if you
> cannot explain a line you cannot submit it. State in your README which sections you
> used it in. This follows the course rule — manual command first, expansion second —
> and using AI in Sections 1 or 2 is an integrity violation, not a style disagreement.

## Setup

Imports and paths only.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm

# The repository root, whether this runs from notebooks/ (Jupyter) or from the root (reproduce.py).
ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(exist_ok=True)

from download_data import MY_PAPER, PAPERS, fetch

## Data

In [ ]:
# Set MY_PAPER once, in download_data.py. fetch() downloads into data/ only if a file is missing.
if MY_PAPER:
    paths = fetch(MY_PAPER)
    print(PAPERS[MY_PAPER]["paper"])
    dwi = pd.read_stata(paths[0])
else:
    print("TODO: set MY_PAPER in download_data.py (castle, card, lee, dw, hansen or abortion).")

  data/hansen_dwi.dta  9.5 MB
Hansen (2015), punishment and deterrence in drunk driving


## Section 1: Positioning (12 pts)

Two pages maximum in the report. What question the paper asks, what was believed before it, what it changed, and how it has been received, with at least three citations to work that engages with it critically. Then the identification strategy and the assumption it rests on, in your own words.

The prose goes in the report. Keep your notes and the three citations here.

**Critical citations (at least three):**

1. de Figueiredo, M. F. P. (2015). Throw away the key or throw away the jail? The effect of punishment on recidivism and social cost. Arizona State Law Journal, 47(4), 1017–1098.
2. Sloan, Eldred, McCutchan, and Platt (2016), “Deterring Rearrests for Drinking and Driving,” Southern Economic Journal 83(2): 416–436
3. Mueller-Smith and Schnepel (2021), “Diversion in the Criminal Justice System,” Review of Economic Studies 88(2): 883–936.

**The identification strategy and its assumption, in my own words:**

Hansen uses a regression discontinuity design to look at drivers just below and above two BAC DUI thresholds, 0.08, and 0.15. He looks at those just above and below these thresholds to test whether harsher penalties have an impact on recidivism in a 4 year window by looking at recidivism rates of those just above and below the thresholds to identify if there is a jump in recidivism. The main assumption it rests on is that other factors affecting recidivism are smooth through the BAC DUI cutoffs. This assumption works because it is impossible for a driver to control their BAC with any precision relative to the cutoffs.

## Section 2: Direct Replication (24 pts)

Reproduce the paper's headline result and report your number beside theirs. You will not match exactly. Document every discrepancy and its likely source: sample construction, variable definition, software defaults, a decision the paper does not describe. Where the paper is ambiguous about a choice, say so: those ambiguities are your Section 3.

In [ ]:
CUTOFF = 0.08
# Converts low_score into the same units as bac
dwi['bac_min'] = dwi['low_score'] / 1000
#Creates age and year dummies and adds the dummy columns to the dataset.
#This is done because Hansen uses age and year fixed effects as controls rather than raw age and year
age_dummy = pd.get_dummies(dwi['aged'], drop_first=True, prefix='Age').astype(int)
year_dummy = pd.get_dummies(dwi['year'], drop_first=True, prefix='Year').astype(int)
dwi[age_dummy.columns] = age_dummy
dwi[year_dummy.columns] = year_dummy
RUNNING = 'bac_min'
H = 0.05
KERNEL = "rectangular"
CONTROLS = ['male', 'white',] + list(age_dummy.columns) + list(year_dummy.columns)
dwi['above_cutoff'] = (dwi[RUNNING] >= CUTOFF).astype(int)

in_band = dwi[np.abs(dwi[RUNNING] - CUTOFF) <= H]
bac_centered = in_band[RUNNING] - CUTOFF

rd_cols = pd.DataFrame(
  {"above_cutoff": in_band['above_cutoff'], "bac_centered": bac_centered,
   "above_centered": in_band['above_cutoff'] * bac_centered}
)
X = pd.concat([rd_cols, in_band[CONTROLS]], axis=1)
X = sm.add_constant(X)

rd_fit = sm.OLS(in_band['recidivism'], X).fit(cov_type='cluster', cov_kwds={'groups': in_band['bac_min']})  #Used Clustered SE as that is what Hansen uses in his regression
JUMP = float(rd_fit.params['above_cutoff'])

no_controls = sm.add_constant(rd_cols)
rd_fit_nocontrol = sm.OLS(in_band['recidivism'], no_controls).fit(cov_type='cluster', cov_kwds={'groups': in_band['bac_min']})  #Used Clustered SE as that is what Hansen uses in his regression
JUMP_nocontrol = float(rd_fit_nocontrol.params['above_cutoff'])


print(JUMP)
print(JUMP_nocontrol)
print(rd_fit.nobs)
print(rd_fit_nocontrol.nobs)



-0.02214472827667978
-0.02184528700936943
93899.0
93899.0


In [ ]:
# Model for narrower threshold
H_NARROW = 0.025
in_band_narrow = dwi[np.abs(dwi[RUNNING] - CUTOFF) <= H_NARROW]
bac_centered_narrow = in_band_narrow[RUNNING] - CUTOFF

rd_cols_narrow = pd.DataFrame(
  {"above_cutoff": in_band_narrow['above_cutoff'], "bac_centered": bac_centered_narrow,
   "above_centered": in_band_narrow['above_cutoff'] * bac_centered_narrow}
)

X_narrow = sm.add_constant(pd.concat([rd_cols_narrow, in_band_narrow[CONTROLS]], axis=1))

rd_fit_narrow = sm.OLS(in_band_narrow['recidivism'], X_narrow).fit(cov_type='cluster', cov_kwds={'groups': in_band_narrow['bac_min']})
JUMP_narrow = float(rd_fit_narrow.params['above_cutoff'])

no_controls_narrow = sm.add_constant(rd_cols_narrow)
rd_fit_narrow_nocontrol = sm.OLS(in_band_narrow['recidivism'], no_controls_narrow).fit(
    cov_type='cluster', cov_kwds={'groups': in_band_narrow['bac_min']})
JUMP_narrow_nocontrol = float(rd_fit_narrow_nocontrol.params['above_cutoff'])

print(JUMP_narrow)
print(JUMP_narrow_nocontrol)
print(rd_fit_narrow.nobs)

-0.019259962204654466
-0.019402093790935646
50016.0


In [ ]:
# Hansen's higher cutoff, 0.15 also ran with no controls and at both bandwidths

CUTOFF2 = 0.15
# Changed filter to directly on low_score to avoid float error at 0.150.
# Also used hard comparison rather than >= as Hansen explicitly makes this a hard cutoff instead.
dwi['above_cutoff2'] = (dwi['low_score'] > 150).astype(int)


in_band2 = dwi[np.abs(dwi['low_score'] - 150) <= 50]
bac_centered2 = in_band2[RUNNING] - CUTOFF2

rd_cols2 = pd.DataFrame(
  {"above_cutoff2": in_band2['above_cutoff2'], "bac_centered2": bac_centered2,
   "above_centered2": in_band2['above_cutoff2'] * bac_centered2}
)
X2 = pd.concat([rd_cols2, in_band2[CONTROLS]], axis=1)
X2 = sm.add_constant(X2)

rd_fit2 = sm.OLS(in_band2['recidivism'], X2).fit(cov_type='cluster', cov_kwds={'groups': in_band2['bac_min']})
JUMP2 = float(rd_fit2.params['above_cutoff2'])

no_controls2 = sm.add_constant(rd_cols2)
rd_fit2_nocontrol = sm.OLS(in_band2['recidivism'], no_controls2).fit(cov_type='cluster', cov_kwds={'groups': in_band2['bac_min']})
JUMP2_nocontrol = float(rd_fit2_nocontrol.params['above_cutoff2'])

in_band2_narrow = dwi[np.abs(dwi['low_score'] - 150) <= 25]
bac_centered2_narrow = in_band2_narrow[RUNNING] - CUTOFF2

rd_cols2_narrow = pd.DataFrame(
  {"above_cutoff2": in_band2_narrow['above_cutoff2'], "bac_centered2": bac_centered2_narrow,
   "above_centered2": in_band2_narrow['above_cutoff2'] * bac_centered2_narrow}
)
X2_narrow = pd.concat([rd_cols2_narrow, in_band2_narrow[CONTROLS]], axis=1)
X2_narrow = sm.add_constant(X2_narrow)

rd_fit2_narrow = sm.OLS(in_band2_narrow['recidivism'], X2_narrow).fit(cov_type='cluster', cov_kwds={'groups': in_band2_narrow['bac_min']})
JUMP2_narrow = float(rd_fit2_narrow.params['above_cutoff2'])

no_controls2_narrow = sm.add_constant(rd_cols2_narrow)
rd_fit2_narrow_nocontrol = sm.OLS(in_band2_narrow['recidivism'], no_controls2_narrow).fit(cov_type='cluster', cov_kwds={'groups': in_band2_narrow['bac_min']})
JUMP2_narrow_nocontrol = float(rd_fit2_narrow_nocontrol.params['above_cutoff2'])

print(JUMP2, JUMP2_nocontrol, rd_fit2.nobs)
print(JUMP2_narrow, JUMP2_narrow_nocontrol, rd_fit2_narrow.nobs)

-0.009019358285274658 -0.009279266904089239 139033.0
-0.00992230991958682 -0.009693339281535141 75909.0


In [ ]:
#Discrepancy Diagnostics
print(JUMP)
print(JUMP_nocontrol)
print(rd_fit.nobs)
print(rd_fit_nocontrol.nobs)
print(len(in_band))
print(in_band['bac1'].count())
print(dwi['bac1'].describe())
in_band_bac1 = dwi[np.abs(dwi['bac1'] - CUTOFF) <= H]
print(len(in_band_bac1))
print(dwi['aged'].min())
print(dwi['year'].unique())
print(in_band['recidivism'].mean())
print(in_band.loc[in_band['above_cutoff'] == 0, 'recidivism'].mean())

-0.02214472827667978
-0.02184528700936943
93899.0
93899.0
93899
93899
count    214558.000000
mean          0.141734
std           0.053901
min           0.000000
25%           0.105000
50%           0.139000
75%           0.176000
max           0.449000
Name: bac1, dtype: float64
90074
21.0
[2007. 2003. 2006. 1999. 2004. 2005. 2001. 2000. 2002.]
0.107434586
0.11707912


### Theirs vs mine

One row per number you replicate. The row label says where the number is in the paper (for example, `Table 2, col 3: effect on log wage`). `gap_pct` is computed for you as 100 × (mine − theirs) / |theirs|. This cell writes `output/headline_table.csv`, which `python reproduce.py` prints and your README quotes.

In [ ]:
headline = pd.DataFrame(
    {
        "paper_estimate": pd.Series(dtype="float"),
        "my_estimate": pd.Series(dtype="float"),
        "gap_pct": pd.Series(dtype="float"),
        "likely_source": pd.Series(dtype="object"),
    },
    index=pd.Index([], name="estimate", dtype="object"),
)
headline.loc["Table 3, Panel A, col 1: BAC >= 0.08, h = 0.05"] = [-0.021, JUMP, np.nan, "public file has ~4% fewer stops in outer window (N -1.3%)"]
headline.loc["Table 3, Panel B, col 1: BAC >= 0.08, h = 0.025"] = [-0.019, JUMP_narrow, np.nan, "within paper's rounding (N +1.3%)"]
headline.loc["Appendix Table 1, Panel A, col 1: BAC >= 0.08, h = 0.05, no controls"] = [-0.020, JUMP_nocontrol, np.nan, "gap persists without controls: not the missing controls"]
headline.loc["Appendix Table 1, Panel B, col 1: BAC >= 0.08, h = 0.025, no controls"] = [-0.019, JUMP_narrow_nocontrol, np.nan, "within paper's rounding"]

headline.loc["Table 4, Panel A, col 1: BAC > 0.15, h = 0.05"] = [-0.010, JUMP2, np.nan, "public file ~5% fewer stops at higher BAC"]
headline.loc["Table 4, Panel B, col 1: BAC > 0.15, h = 0.025"] = [-0.011, JUMP2_narrow, np.nan, "public file ~3.5% fewer stops"]
headline.loc["Appendix Table 2, Panel A, col 1: BAC > 0.15, h = 0.05, no controls"] = [-0.011, JUMP2_nocontrol, np.nan, "gap persists without controls"]
headline.loc["Appendix Table 2, Panel B, col 1: BAC > 0.15, h = 0.025, no controls"] = [-0.011, JUMP2_narrow_nocontrol, np.nan, "gap persists without controls"]
# One line per row; take my_estimate from the cell that computed it, never type it in.
# headline.loc["Table 2, col 3: <what it measures>"] = [<paper's number>, <your variable>, np.nan, "<likely source>"]

headline["gap_pct"] = 100 * (headline["my_estimate"] - headline["paper_estimate"]) / headline["paper_estimate"].abs()
headline.to_csv(ROOT / "output" / "headline_table.csv")
headline

,paper_estimate,my_estimate,gap_pct,likely_source
estimate,,,,
"Table 3, Panel A, col 1: BAC >= 0.08, h = 0.05",-0.021,-0.022145,-5.451087,public file has ~4% fewer stops in outer windo...
"Table 3, Panel B, col 1: BAC >= 0.08, h = 0.025",-0.019,-0.019260,-1.368222,within paper's rounding (N +1.3%)
"Appendix Table 1, Panel A, col 1: BAC >= 0.08, h = 0.05, no controls",-0.020,-0.021845,-9.226435,gap persists without controls: not the missing...
"Appendix Table 1, Panel B, col 1: BAC >= 0.08, h = 0.025, no controls",-0.019,-0.019402,-2.116283,within paper's rounding
"Table 4, Panel A, col 1: BAC > 0.15, h = 0.05",-0.010,-0.009019,9.806417,public file ~5% fewer stops at higher BAC
"Table 4, Panel B, col 1: BAC > 0.15, h = 0.025",-0.011,-0.009922,9.797183,public file ~3.5% fewer stops
"Appendix Table 2, Panel A, col 1: BAC > 0.15, h = 0.05, no controls",-0.011,-0.009279,15.643028,gap persists without controls
"Appendix Table 2, Panel B, col 1: BAC > 0.15, h = 0.025, no controls",-0.011,-0.009693,11.878734,gap persists without controls


### Discrepancy log

One row per discrepancy, including the ones you could not explain. A clean match reported without investigation is less credible than a 3% gap that is explained.

| # | Estimate (row above) | What differs | Size of the gap | Likely source | How I checked it | Is the paper ambiguous here? |
|---|---|---|---|---|---|---|
| 1 | 1-4 | RD estimate is slightly off at both thresholds and bandwidths | -5.45% on the main regression | sample construction | First checked if my estimate was outside rounding error and it was. Then checked to see if there were any dropped rows. Checked to see if this was a result of not having access to same controls, but rd without controls did not match the paper estimate. Then checked to see if the mean recidivism below the cutoff was the same (17%) and it was. Likely leaves difference in N as the reason| No, the running variable, bandwidth, kernel, form, and clustering are all clearly stated. |
| 2 |5-8 |Estimates at the higher BAC cutoff are off |Between 10-16% |Same checks as above |Sample construction, possible paper coding |Somewhat. The table 1 intervals are clear on the hard vs soft cutoff comparison but the text sometimes uses "above" to refer to both thresholds when it means at or above |
| 3 |All rows|Mismatched sample size |At 0.08 cutoff, +1.3% at H=0.025 and -1.3% at H=0.05. At 0.15 cutoff, -3.5% at H=0.025, -5.2% at H-0.05 |Sample construction |Checked if the regressions window counts matched their N. Checked that minimum age is 21 and years are 1999-2007. Checked N at 0.08 cutoff and H=0.05. |Yes. The paper does not give the size of its 1999-2007 sample. Also, for the H=0.05 cutoff, the paper table 4 lists N=146,626, but in Appendix Table 4 is listed as 146,426. Table 4 subgroup N sums to 146,426 supporting appendix number. |

**Ambiguities in the paper, to take into Section 3:**

- Treatment rule at 0.15, and whether it is actually a > or >= comparison